# Chapter 4, Exercise 5: Clitic segmentation of وللمكتبة with CAMeL Tools (and Stanza)

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 4, Exercise 5.** Assume a simple clitic-stripping analyzer that removes leading conjunction, article, and preposition clitics and trailing pronoun clitics. Segment وللمكتبة (wa-lil-maktaba, 'and for the library') into its clitics and stem, then explain how this segmentation reduces OOV forms compared with word-level tokenization. State your assumptions about which clitics the analyzer handles. Then run a specified tokenizer (for example CAMeL Tools or Stanza) on this exact word and compare its output with your manual segmentation, reporting the tool name, version, model, and segmentation scheme you used.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.

## Testing status

The CAMeL Tools cells were executed. The Stanza cell is optional (`RUN_STANZA`) and was executed separately in the same environment; its output is quoted in the text.


## 1. Manual segmentation with the assumed simple analyzer

Assumptions: the analyzer strips, from the left, one conjunction (و or ف), then one preposition (ب, ل, ك), then the article ال; from the right, one pronoun enclitic (ـه, ـها, ـهم, ـك, ـي, ...). It does not touch inflectional affixes (ـة, ـات, ـون) and it does not know that ل + ال is written لل (the alif of the article is dropped after ل).

| Piece | Type | Gloss |
|---|---|---|
| وَ wa- | conjunction proclitic | and |
| لِ li- | preposition proclitic | for, to |
| الـ al- | definite article proclitic (written as a single ل after لِ) | the |
| مَكْتَبَة maktaba | stem (noun) | library |
| (none) | pronoun enclitic | |

Manual result: **و + ل + ال + مكتبة**, four pieces. Note the orthographic rule that the analyzer must know: لِ + الـ is written لل, so the surface string contains only one alif-less lām sequence; a naive prefix stripper that looks for the literal string ال would fail on this word.

In [1]:
!pip install -q camel-tools
!camel_data -i disambig-mle-calima-msa-r13

No new packages will be installed.


In [2]:
import camel_tools
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.morphological import MorphologicalTokenizer

WORD = "وللمكتبة"
print("CAMeL Tools version:", camel_tools.__version__)
mle = MLEDisambiguator.pretrained("calima-msa-r13")   # model: MLE disambiguator over CALIMA-MSA r13

for scheme in ["atbtok", "d3tok", "bwtok"]:
    tok = MorphologicalTokenizer(disambiguator=mle, scheme=scheme, split=True)
    print(f"scheme={scheme:7s} ->", tok.tokenize([WORD]))

# Show the chosen analysis so the segmentation can be read off the Buckwalter tag
a = mle.disambiguate([WORD])[0].analyses[0].analysis
print("\ndiac:", a["diac"]); print("bw:  ", a["bw"]); print("lemma:", a["lex"], "| pos:", a["pos"], "| gloss:", a["gloss"])

CAMeL Tools version: 1.6.0


scheme=atbtok  -> ['و+', 'ل+', 'المكتبة']
scheme=d3tok   -> ['و+', 'ل+', 'ال+', 'مكتبة']
scheme=bwtok   -> ['و+', 'ل+', 'ال+', 'مكتب', '+ة']

diac: وَلِلمَكْتَبَة
bw:   وَ/CONJ+لِ/PREP+ال/DET+مَكْتَب/NOUN+َة/NSUFF_FEM_SG
lemma: مَكْتَبَة | pos: noun | gloss: and_+_to;for_+_the+library;bookstore+[fem.sg.]


## 2. Comparison with the manual segmentation

| Source | Output for وللمكتبة | Pieces |
|---|---|---|
| Manual (assumed analyzer) | و + ل + ال + مكتبة | 4 |
| CAMeL Tools 1.6.0, MLE disambiguator, `calima-msa-r13`, scheme **D3** (`d3tok`) | و+ ل+ ال+ مكتبة | 4 (identical to the manual segmentation) |
| Same, scheme **ATB** (`atbtok`) | و+ ل+ المكتبة | 3 (the Penn Arabic Treebank scheme keeps the article attached to the noun) |
| Same, scheme **BW** (`bwtok`) | و+ ل+ ال+ مكتب +ة | 5 (splits the feminine suffix ة as well) |

The D3 scheme matches the assumed analyzer exactly; ATB and BW differ because they draw the line between "clitic" and "affix" elsewhere. This is why the exercise asks for the **scheme** to be reported: three correct outputs of one tool give three different token counts.

## 3. Optional cross-check with Stanza

Stanza (Universal Dependencies, PADT model) performs multiword-token expansion. It needs sentence context to split reliably; in isolation the single word may be returned unsplit. Set `RUN_STANZA = True` to run it (downloads about 400 MB).

In [3]:
RUN_STANZA = False
if RUN_STANZA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stanza"], check=True)
    import stanza
    stanza.download("ar", verbose=False)
    nlp = stanza.Pipeline("ar", processors="tokenize,mwt,pos,lemma", verbose=False, use_gpu=False)
    print("stanza", stanza.__version__, "model: ar (PADT)")
    for text in [WORD, "ذهب الطلاب إلى المدرسة وللمكتبة أيضا"]:
        doc = nlp(text)
        for s in doc.sentences:
            for tok in s.tokens:
                if tok.text == WORD:
                    print(f"in '{text}': {WORD} ->", [(w.text, w.upos, w.lemma) for w in tok.words])

When tested (Stanza 1.14.0, Arabic PADT model, CPU), the word in sentence context was expanded to **و (CCONJ) + ل (ADP) + المكتبة (NOUN, lemma مَكتَبَة)**, that is, the ATB-style three-piece segmentation with the article attached; in isolation it came back unsplit. Stanza is a UD tokenizer and tagger, not a full morphological analyzer (Section 4.4.3), so it should be reported as such.

## 4. Why segmentation reduces OOV forms

A word-level vocabulary needs a separate entry for every surface string. The stem مكتبة alone can appear as مكتبة, المكتبة, للمكتبة, وللمكتبة, بالمكتبة, ومكتبتها, كمكتبة, ... and each of these is a distinct, individually rare word type. After clitic stripping the language model and the recognizer see the same unit مكتبة in every one of them, plus a handful of very frequent clitic tokens (و, ل, ال, ب, ك, ـها, ...). The number of types collapses toward the number of stems, the counts per type rise, and a form never seen in training (say فبمكتبتهم) is no longer OOV as long as its stem and its clitics were each seen separately. The price is an analyzer that knows the orthographic rules (لل for لِ + الـ, the pronoun after ة becoming ت) and, for dialects, a spelling convention it can rely on (Section 5.7.2).